# Phase Space Plot Speed Benchmark
Goal: test notebook-only speedups without modifying src/.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from src.integrator.integrate import FastSitnikovSimulation

In [ ]:
def phase_space_plot_baseline(e, N_v, N_t, N_it, random_dist=False, spacing=0):
    if N_v <= 0 or N_t <= 0 or N_it <= 0:
        raise ValueError("N_v, N_t, and N_it must be positive")
    if spacing < 0:
        raise ValueError("spacing must be non-negative")

    sim = FastSitnikovSimulation(e=e)
    period = 2.0 * np.pi
    v_max = 2.0 / np.sqrt(1.0 - e)

    if random_dist:
        rng = np.random.default_rng()
        v_samples = np.sort(rng.uniform(0.0, v_max, size=int(N_v)))
        t_samples = np.sort(rng.uniform(0.0, period, size=int(N_t)))
    else:
        v_samples = np.linspace(0.0, v_max, int(N_v) + 2)[1:-1]
        t_samples = np.linspace(0.0, period, int(N_t), endpoint=False)

    plotted_points = np.empty((0, 2), dtype=float)
    kept_points_t = []
    kept_points_v = []

    for t0 in t_samples:
        for v0 in v_samples:
            trajectory = [(float(v0), float(t0))]
            v_curr = float(v0)
            t_curr = float(t0)

            for _ in range(int(N_it)):
                v_next, t_next = sim.phi_fast(v=v_curr, t=t_curr, return_mod_period=True)
                if v_next is None or t_next is None:
                    break
                trajectory.append((float(v_next), float(t_next)))
                v_curr = float(v_next)
                t_curr = float(t_next)

            if len(trajectory) < int(N_it + 1):
                continue

            trajectory_arr = np.asarray(trajectory, dtype=float)
            if spacing > 0 and plotted_points.size > 0:
                delta_t = np.abs(trajectory_arr[:, None, 1] - plotted_points[None, :, 1])
                delta_t = np.minimum(delta_t, period - delta_t)
                delta_v = trajectory_arr[:, None, 0] - plotted_points[None, :, 0]
                distances = np.sqrt(delta_v * delta_v + delta_t * delta_t)
                if np.any(np.min(distances, axis=1) < spacing):
                    continue

            kept_points_v.extend(trajectory_arr[:, 0])
            kept_points_t.extend(trajectory_arr[:, 1])
            plotted_points = np.vstack((plotted_points, trajectory_arr))

    fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={"projection": "polar"})
    if kept_points_t:
        ax.scatter(kept_points_t, kept_points_v, s=1, color="black")
    ax.set_rlim(0.0, v_max)
    return fig, ax

In [ ]:
def phase_space_plot_fast(e, N_v, N_t, N_it, random_dist=False, spacing=0, phi_time_window=4.0 * np.pi):
    if N_v <= 0 or N_t <= 0 or N_it <= 0:
        raise ValueError("N_v, N_t, and N_it must be positive")
    if spacing < 0:
        raise ValueError("spacing must be non-negative")

    sim = FastSitnikovSimulation(e=e, phi_time_window=phi_time_window)
    period = 2.0 * np.pi
    v_max = 2.0 / np.sqrt(1.0 - e)

    if random_dist:
        rng = np.random.default_rng()
        v_samples = np.sort(rng.uniform(0.0, v_max, size=int(N_v)))
        t_samples = np.sort(rng.uniform(0.0, period, size=int(N_t)))
    else:
        v_samples = np.linspace(0.0, v_max, int(N_v) + 2)[1:-1]
        t_samples = np.linspace(0.0, period, int(N_t), endpoint=False)

    plotted_points = np.empty((0, 2), dtype=float)
    kept_points_t = []
    kept_points_v = []

    for t0 in t_samples:
        for v0 in v_samples:
            trajectory = [(v0, t0)]
            v_curr = v0
            t_curr = t0

            for _ in range(int(N_it)):
                v_next, t_next = sim.phi_fast(v=v_curr, t=t_curr, return_mod_period=True)
                if v_next is None or t_next is None:
                    break
                trajectory.append((v_next, t_next))
                v_curr, t_curr = v_next, t_next

            if len(trajectory) < int(N_it + 1):
                continue

            trajectory_arr = np.asarray(trajectory, dtype=float)
            if spacing > 0 and plotted_points.size > 0:
                delta_t = np.abs(trajectory_arr[:, None, 1] - plotted_points[None, :, 1])
                delta_t = np.minimum(delta_t, period - delta_t)
                delta_v = trajectory_arr[:, None, 0] - plotted_points[None, :, 0]
                distances = np.sqrt(delta_v * delta_v + delta_t * delta_t)
                if np.any(np.min(distances, axis=1) < spacing):
                    continue

            kept_points_v.extend(trajectory_arr[:, 0])
            kept_points_t.extend(trajectory_arr[:, 1])
            plotted_points = np.vstack((plotted_points, trajectory_arr))

    fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={"projection": "polar"})
    if kept_points_t:
        ax.scatter(kept_points_t, kept_points_v, s=1, color="black")
    ax.set_rlim(0.0, v_max)
    return fig, ax

In [ ]:
def benchmark(func, kwargs, repeats=2):
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fig, ax = func(**kwargs)
        dt = time.perf_counter() - t0
        times.append(dt)
        plt.close(fig)
    return float(np.median(times))

params = dict(e=0.5, N_v=20, N_t=6, N_it=50, random_dist=False, spacing=0)
t_base = benchmark(phase_space_plot_baseline, params, repeats=2)
t_fast = benchmark(phase_space_plot_fast, params, repeats=2)
speedup = t_base / t_fast
print(f"baseline: {t_base:.3f}s")
print(f"optimized: {t_fast:.3f}s")
print(f"speedup: {speedup:.2f}x")

In [ ]:
fig, ax = phase_space_plot_fast(e=0.5, N_v=20, N_t=6, N_it=50, spacing=0)
plt.show()